In [41]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool
from shared.model_params import shift, lag_size, n_clusters
import sqlite3

In [42]:
from sklearn.model_selection import train_test_split, TimeSeriesSplit, LeaveOneGroupOut, GroupKFold
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, mean_squared_error, silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.base import TransformerMixin, BaseEstimator, RegressorMixin
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [43]:
import torch
import torch.nn as nn

In [44]:
class DataLoader:
    def __init__(self, countries, max_count = (0, 5), shift_size = 4, lag_size = 6, logging = True):
        self.countries = countries
        self.max_count = max_count
        self.shift_size = shift_size
        self.lag_size = lag_size
        self.logging = logging
        self.conn = sqlite3.connect("dataset.db")

    def get_datasets(self):
        data = pd.read_sql(f"SELECT * FROM runs_stats WHERE game_ind BETWEEN {self.max_count[0]} AND {self.max_count[1]}\
                           AND country IN {tuple(self.countries)}", self.conn)
        data["war_status"] = data["war_status"].astype(int)
        return self._split_Xy(data)
    
    def _split_Xy(self, data):
        X = data.drop(["gdp"], axis = 1)[:-self.shift_size]
        y = data["gdp"][self.shift_size:]
        return X, y

    
    def prepare_target(self, y):
        return np.log1p(y)
    
    @staticmethod
    def _generate_period_names(data):
        bins = [0, 1850, 1862, 1873, 1885, 1895, 1906, np.inf]
        labels = [
            '1836-1850',
            '1851-1862', 
            '1863-1873', 
            '1874-1885', 
            '1886-1895', 
            '1896-1906', 
            '1907+'
        ]
        year = pd.to_datetime(data["date"]).dt.year
        name = data["flag_name"].str.split("_").str[0]
        periods = pd.cut(year, bins=bins, labels=labels)
        return name + " (" + periods.astype(str) + ")"

    
    def add_name(self, data):
        data["name"] = self._generate_period_names(data)
        return data

    def add_new_cols(self, arr):
        res = []
        for col in arr:
            for i in range(1, self.lag_size + 1):
                res.append(f"{col}_log_diff_{i}")
            res.append(f"{col}_rolling_mean_3")
            res.append(f"{col}_rolling_std_5")
        return res
    
    def cluster_cols(self, num):
        res = []
        for i in range(num):
            res.append(f"dist_to_c{i+1}")
        return res

In [45]:
class DataProcessor(BaseEstimator, TransformerMixin):
    def __init__(self, num_cols, important_cols, lag_size):
        self.num_cols = num_cols
        self.important_cols = important_cols
        self.lag_size = lag_size

    def fit(self, X, y=None):
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X = X.copy()
        new_cols = {}
        X["gdp"] = X["gdp"].bfill().ffill()
        X["gdp"] = np.log1p(X["gdp"])
        X["tax_progressivity"] = X["rich_tax"] - X["poor_tax"]
        X["innovation_total"] = X["prom_innov"] + X["comm_innov"]
        for col in self.num_cols:
            X[col] = X[col].bfill().ffill().astype(float)
        for col in self.important_cols:
            for i in range(1, self.lag_size + 1):
                new_cols[f"{col}_log_diff_{i}"] = np.log1p(X[col]).diff(i)
            new_cols[f"{col}_rolling_mean_3"] = X[col].rolling(3).mean()
            new_cols[f"{col}_rolling_std_5"] = X[col].rolling(5).std()
        X = pd.concat([X, pd.DataFrame(new_cols, index=X.index)], axis=1)
        return X.ffill().bfill()
    
class DataSelector(BaseEstimator, TransformerMixin):
    def __init__(self, all_cols):
        self.all_cols = all_cols
    
    def fit(self, X, y = None):
        return self
    
    def transform(self, X):
        return X[self.all_cols]
    

class ClusterPredictor(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=5, features_for_clustering=None):
        self.n_clusters = n_clusters
        self.features_for_clustering = features_for_clustering
        self.scaler = StandardScaler()
        self.kmeans = KMeans(n_clusters=self.n_clusters, random_state=42, n_init=10)

    def fit(self, X, y=None):
        x_subset = X[self.features_for_clustering]
        x_scaled = self.scaler.fit_transform(x_subset)
        self.kmeans.fit(x_scaled)
        return self

    def transform(self, X):
        X = X.copy()
        x_subset = X[self.features_for_clustering]
        x_scaled = self.scaler.transform(x_subset)
        cluster_distances = self.kmeans.transform(x_scaled)
        df_clust = pd.DataFrame(
            cluster_distances, 
            columns=[f"dist_to_c{i + 1}" for i in range(self.n_clusters)],
            index=X.index
        )
        X["cluster"] = self.kmeans.predict(x_scaled)
        X["cluster"] = X["cluster"].astype(str)

        X = pd.concat([X, df_clust], axis=1)
        
        return X

    def get_scaler(self):
        return self.scaler

In [46]:
loader = DataLoader(["ENG", "RUS", "AUS", "CHI", "BEL", "PRU", "SWE", "FRA", "ETH", "TUR", "SIC", "MEX", "AFG", "PER", "POR", "USA"], (1, 6), shift_size=shift, lag_size=lag_size, logging=False)

In [47]:
cat_cols = ["most_popular_party", "war_status", "cluster"]
num_cols = ["money_activity", "subside_percent", "rentability", "diversification", "gdp_per_cap", "gdp_per_reg",\
             "country_savings", "bank_savings", "population_savings", "money_mass", "all_employemenent", "all_free_work_places", "fabric_worker_salary",\
             "capitalist_salary", "literacy", "military_budget", "naval_budget", "army_innov", "naval_innov", "country_size", "population_per_reg",\
             "rich_tax", "middle_tax", "poor_tax", "prom_innov", "comm_innov", "tax_progressivity", "innovation_total"]

important_cols = ["gdp", "poor_tax", "bank_savings", "population"]
im_cols = ["poor_tax", "bank_savings", "population"]
new_num_cols = loader.add_new_cols(im_cols)
deleted_cols = ["industrial_level", "subside_pct", "gold_income", "goverement"]
all_cols = cat_cols + num_cols + new_num_cols + loader.cluster_cols(3)
cluster_cols = ["money_activity", "rentability", "diversification", "literacy", "prom_innov", "comm_innov", "innovation_total", "tax_progressivity", "subside_percent", "fabric_worker_salary",\
                "capitalist_salary", "population_per_reg"]

In [48]:
X, y = loader.get_datasets()